# SPINE-GPE v7.1 — FASE 0: DATA, IDENTIFIABILITY & REPRODUCIBILITY LOCK

**Objetivo:** Inventário automatizado, hash de arquivos, schema registry, contratos de dados e DAGs de identificação.

**Status Epistêmico:** Tier A (Reprodutibilidade e Identificação)

---

In [ ]:
# CONFIGURAÇÃO INICIAL E MONTAGEM DO GDRIVE
import os
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime

# Detectar ambiente
try:
    from google.colab import drive
    IN_COLAB = True
    print("🔵 Ambiente: Google Colab detectado")
    drive.mount('/content/drive')
    GDRIVE_ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
except ImportError:
    IN_COLAB = False
    print("🔵 Ambiente: Local/Outro")
    GDRIVE_ROOT = Path(os.environ.get('GDRIVE_ROOT', './data'))

# Paths críticos
DATA_RAW = GDRIVE_ROOT / '01_raw'
DATA_FROZEN = GDRIVE_ROOT / '05_frozen'
REPORTS_DIR = GDRIVE_ROOT / '06_reports'
MANIFESTS_DIR = GDRIVE_ROOT / '04_manifests'

print(f"📂 GDrive Root: {GDRIVE_ROOT}")
print(f"📂 Raw Data: {DATA_RAW}")
print(f"📂 Frozen Data: {DATA_FROZEN}")

In [ ]:
# FUNÇÃO DE HASH SHA-256 PARA INTEGRIDADE
def compute_sha256(file_path: Path) -> str:
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

# INVENTÁRIO AUTOMATIZADO DE FONTES
def inventory_sources(base_path: Path) -> dict:
    inventory = {'timestamp': datetime.utcnow().isoformat(), 'sources': []}
    for pattern in ['*.parquet', '*.json', '*.csv']:
        for file_path in base_path.rglob(pattern):
            if '.git' not in str(file_path):
                file_stat = file_path.stat()
                inventory['sources'].append({
                    'path': str(file_path.relative_to(base_path)),
                    'size_bytes': file_stat.st_size,
                    'hash_sha256': compute_sha256(file_path) if file_stat.st_size > 0 else 'EMPTY',
                    'modified': datetime.fromtimestamp(file_stat.st_mtime).isoformat()
                })
    return inventory

print("✅ Funções de inventário e hash carregadas.")

In [ ]:
# EXECUTAR INVENTÁRIO
if DATA_RAW.exists():
    print(f"🔍 Executando inventário em {DATA_RAW}...")
    inventory = inventory_sources(DATA_RAW)
    print(f"📊 Fontes encontradas: {len(inventory['sources'])}")
    inv_file = REPORTS_DIR / 'inventory' / f"source_inventory_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    inv_file.parent.mkdir(parents=True, exist_ok=True)
    with open(inv_file, 'w') as f:
        json.dump(inventory, f, indent=2)
    print(f"💾 Inventário salvo em: {inv_file}")
else:
    print(f"⚠️ Pasta RAW não encontrada em {DATA_RAW}")

In [ ]:
# SCHEMA REGISTRY OFICIAL
OFFICIAL_SCHEMA = {
    'RAIS': {'required_columns': ['ano', 'cbo', 'municipio', 'salario_base'], 'epistemic_tier': 'A_OBSERVED'},
    'PNADc_Plataforma': {'required_columns': ['S140093', 'SD14001', 'VD3003', 'peso'], 'epistemic_tier': 'A_OBSERVED_DIRECT'},
    'PNADc_Backcast': {'required_columns': ['prob_platform', 'classification'], 'epistemic_tier': 'C_MODEL_PROXY'}
}
print("✅ Schema Registry Oficial carregado.")
for src, sch in OFFICIAL_SCHEMA.items():
    print(f"   - {src}: Tier {sch['epistemic_tier']}")

In [ ]:
# GOLDEN TESTS
def run_golden_tests(frozen_path: Path) -> dict:
    results = {'passed': 0, 'failed': 0, 'details': []}
    critical_files = ['certified_pnadc_platform_2022.parquet', 'certified_pnadc_platform_2024.parquet', 'rais_formal_baseline.parquet']
    for fname in critical_files:
        fpath = frozen_path / fname
        if fpath.exists():
            results['passed'] += 1
            results['details'].append({'test': f'Exists:{fname}', 'status': 'PASS'})
        else:
            results['failed'] += 1
            results['details'].append({'test': f'Exists:{fname}', 'status': 'FAIL'})
    return results

if DATA_FROZEN.exists():
    print(f"🧪 Executando Golden Tests em {DATA_FROZEN}...")
    test_results = run_golden_tests(DATA_FROZEN)
    print(f"✅ Passos: {test_results['passed']} | ❌ Falhas: {test_results['failed']}")
else:
    print(f"⚠️ Pasta FROZEN não encontrada em {DATA_FROZEN}")
    test_results = {'passed': 0, 'failed': len(['a','b','c'])}

In [ ]:
# DAG DE IDENTIFICAÇÃO
print("""
DAG DE IDENTIFICAÇÃO CAUSAL — SPINE-GPE v7.1
============================================
Nós: Y(Renda), D(Plataforma), X(Covariáveis), Z(Instrumentos)
Relações: X→D, X→Y, D→Y, Z→D, Z↛Y
LIMITES:
  ✅ Tier A: Associação D-Y ajustada por X (TMLE)
  ⚠️  Tier B: Decomposição Oaxaca
  ❌ Tier C/BLOCKED: IV/GMM (Instrumento fraco)
  ❌ Tier D: Mecanismos Algorítmicos Diretos (Não observados)
""")

In [ ]:
# FECHAMENTO FASE 0
phase0_closure = {
    'phase': '0_DATA_LOCK',
    'timestamp': datetime.utcnow().isoformat(),
    'inventory_count': len(inventory.get('sources', [])) if 'inventory' in locals() else 0,
    'golden_tests_passed': test_results['passed'] if 'test_results' in locals() else 0,
    'golden_tests_failed': test_results['failed'] if 'test_results' in locals() else 0,
    'schema_version': 'v7.1.0',
    'identification_status': 'PARTIAL (IV_BLOCKED)',
    'ready_for_phase1': True if DATA_FROZEN.exists() else False
}
closure_file = MANIFESTS_DIR / 'phase0_closure_manifest.json'
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
with open(closure_file, 'w') as f:
    json.dump(phase0_closure, f, indent=2)
print(f"\n✅ FASE 0 CONCLUÍDA! Manifesto salvo em: {closure_file}")
print(f"🔓 Pronto para Fase 1: {phase0_closure['ready_for_phase1']}")